# Automatic Deep Research - Building more reliable systems

Welcome to the first practice lab of this module! 

In the last module, you went through a very interesting use case of multi agent systems: building your custom deep research system. In this lab you will use what you have already built in Module 1, and add decision making tools, like execution hooks and guardrails, as well as memory to improve the reliability of your crew.

**What you'll learn:**
- How to add programmatic guardrails to make your multi agent system more robust
- How to add execution hooks to inject logic after the agents run
- How to add memory to your crew

## Background

As a research consultant, you're constantly tasked with producing comprehensive reports on diverse topics for demanding clients. You need to build an AI research crew that can rapidly gather, verify, and synthesize information from across the internet, delivering reliable, fact-checked reports that meet tight deadlines and exacting standards regardless of the subject matter.

## General instructions
In this lab you will be presented with a structure of the code, but you will need to complete some of it. 

To successfully run this lab, replace all instances of the placeholder `None` with your own code. Sections where you need to write code will be delimited between `### START CODE HERE ###` and `### END CODE HERE ###`.

If you are stuck, or simply want to copy a solution into your notebook so that you can execute it, you can find all solution code inside the [Solution](Solution) folder.

**<font color='#5DADEC'>Please make sure to save your work periodically, so you don't lose any progress.</font>**

## Table of contents

- [1. Problem statement](#1)
- [2. Set up your notebook](#2)
- [3. Agents](#3)
- [4. Guardrails](#4)
- [5. Tasks](#5)
- [6. Execution hooks](#6)
- [7. Crew](#7)
  - [7.1. Define the crew](#7-1)
  - [7.2. Define the inputs](#7-2)
  - [7.3. Run the crew](#7-3)

<a id="1"></a>

## 1. Problem statement

The goal of this lab is to take a multi-agent system that can interpret a user's input, and create an action plan, then do the actual research and fact checking, and finally output a report you can share with the client. In order to make the output more reliable, you will add new  guardrails, execution hooks and memory into strategic elements of Crew. 

Here is a visual summary of the structure of your crew, as well as the new elements you will be adding: 

<img src="../images/lab1-agents-tasks-diagram.PNG">


<a id="2"></a>

## 2. Set up your notebook

Begin by setting up the notebook by importing all necessary modules, and configuring the environment variables so you can connect to OpenAI.

In [1]:
# Patch to disable SSL verification for Coursera
from patch import disable_ssl_verification
disable_ssl_verification()

from crewai import Agent, Task, Crew
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
import os
os.environ["CREWAI_TESTING"] = "true"
from utils import get_openai_api_key, get_exa_api_key
from IPython.display import Markdown
import yaml

# set the OpenAI model (gpt-4o-mini)
os.environ["MODEL"] = "gpt-4o-mini"
# set up the OpenAI API key 
os.environ["OPENAI_API_KEY"] = get_openai_api_key()
# set the EXA API key
os.environ["EXA_API_KEY"] = get_exa_api_key()

<a id="3"></a>

## 3. Agents

For this system, you will use four agents:
- Research Planner
- Internet Researcher
- Fact checker
- Report Writer

All their arguments (`role`, `goal`, `backstory`) are already given to you, and given in a YAML file you can use to import the configuration. If you want to take a closer look, open the [config/agents.yaml](config/agents.yaml) file in the file navigator on the left.

In the labs, we have added two parameters not shown in the demo videos: `max_rpm`, and `max_iter`. `max_rpm` sets the maximum requests per minute to avoid rate limits, while `max_iter` limits the maximum iterations before the agent must provide its best answer. Setting these two parameters helps make the agents run a little faster, so the lab doesn't take as long to complete. 

Run the next cell to create an instance of each agent, as well as the tools for the **Internet Researcher** agents.

In [2]:
# create the tool instances
exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL")) 
scrape_website_tool = ScrapeWebsiteTool()

# load the configuration file for the agents
with open('config/agents.yaml', 'r') as file:
        agent_config = yaml.safe_load(file)

# create the agents using the configuration
research_planner = Agent(
        config=agent_config['research_planner'],
        verbose=True,
        max_rpm=150,
        max_iter=15
        )
internet_researcher = Agent(
        config=agent_config['internet_researcher'],
        tools=[exa_search_tool, scrape_website_tool],
        verbose=True,
        max_rpm=150,
        max_iter=15
        )
fact_checker = Agent(
        config=agent_config['fact_checker'],
        tools=[exa_search_tool, scrape_website_tool],
        verbose=True,
        max_rpm=150,
        max_iter=15
        )
report_writer = Agent(
        config=agent_config['report_writer'],
        verbose=True,
        max_rpm=150,
        max_iter=15
        )

<a id="4"></a>

## 4. Guardrails

To make your system more robust, you want to add guardrails to your tasks. These guardrails provide a way to validate and transform task outputs before they are passed to the next task, helping ensure data quality and providing feedback to agents when their output doesn't meet specific criteria. You can find out more about guardrails in the [docs](https://docs.crewai.com/en/concepts/tasks#task-guardrails).

In this lab, you will be working with [**Task Guardrails**](https://docs.crewai.com/en/concepts/tasks#task-guardrails). These are custom functions that check if a task's output meets your requirements before passing it to the next task. They help ensure quality and give feedback to agents when their work needs improvement.

The guardrail functions must accept exactly one parameter (the task output they are reviewing), and should return a tuple of `(bool, Any)`. If the validation is successful, it returns a tuple of `(bool, Any)`. For example: (True, validated_result). If it fails, it needs to return a tuple of `(bool, str)`. For example: (False, "Error message explaining the failure"). For more information, you can check out the [docs](https://docs.crewai.com/en/concepts/tasks#task-guardrails).

In particular, you will implement a guardrail for the final output. You want to make sure the final report has all the sections needed: 
- Summary
- Insights (or recommendations)
- Citations (or References)

To make sure the keywords are in fact in a section title, you should check the line begins with `#`. You will use regular expressions for that.

In [3]:
import re

# write the custom guardrail function
def write_report_guardrail(output):
    # get the raw output from the TaskOutput object
    try:
        output = output if type(output)==str else output.raw 
    except Exception as e:
        return (False, ("Error retrieving the `raw` argument: "
                        f"\n{str(e)}\n"
                        )
                )
    
    # convert the output to lowercase
    output_lower = output.lower()

    # check that the summary section exists
    if not re.search(r'#+.*summary', output_lower):
        return (False, 
                "The report must include a Summary section with a header like '## Summary'"
                )

    # check that the insights or recommendations sections exist
    if not re.search(r'#+.*insights|#+.*recommendations', output_lower):
        return (False, 
                "The report must include an Insights section with a header like '## Insights'"
                )

    ### START CODE HERE ###

    # check that the citations (or references) section exists
    if not re.search(r'#+.*citations|#+.*references', output_lower):
        return (False, 
                "The report must include a Citations or References section with a header like '## Citations' or '## References' "
                )
        
    ### END CODE HERE ###
    return (True, output)

Run the next two cells to test the guardrail function, one cell has a valid structure, and the other is missing sections

In [4]:
test_report_pass = """
# Report title

## Executive Summary
This is a summary.

## Insights
These are the insights.

## Citations
1. Citation 1
2. Citation 2
"""

write_report_guardrail(test_report_pass)

(True,
 '\n# Report title\n\n## Executive Summary\nThis is a summary.\n\n## Insights\nThese are the insights.\n\n## Citations\n1. Citation 1\n2. Citation 2\n')

In [5]:
test_report_fail = """
# Report title

## Executive Summary
This is a summary.
"""

write_report_guardrail(test_report_fail)

(False,
 "The report must include an Insights section with a header like '## Insights'")

<a id="5"></a>

## 5. Tasks
Now you are ready to create the tasks. Just as you did with the agents, you will load the configuration from a YAML file. You can find it in [`config/tasks.yaml`](config/tasks.yaml). 
In this case, you will need to add the agents, and the guardrails you just created to the corresponding tasks.

In [6]:
# load the configuration file for the tasks
with open('config/tasks.yaml', 'r') as file:
    task_config = yaml.safe_load(file)

### START CODE HERE ###

# create the tasks using the configuration
create_research_plan = Task( 
    config=task_config['create_research_plan'],
    agent=research_planner,
)

gather_research_data = Task( 
    config=task_config['gather_research_data'],
    agent=internet_researcher,
)

verify_information_quality = Task( 
    config=task_config['verify_information_quality'],
    agent=fact_checker,
)

write_final_report = Task( 
    config=task_config['write_final_report'],
    agent=report_writer,
    guardrails=[write_report_guardrail], # add the custom guardrail
)

### END CODE HERE ###

<a id="6"></a>

## 6. Execution hooks

The last step before creating the Crew is creating an [after kickoff hook](https://docs.crewai.com/en/learn/before-and-after-kickoff-hooks#after-kickoff-hook). This is a function that will execute after your crew has finished all the tasks. These functions receive a result object, which contains the outputs of the crew's execution.

In this case, you will create a hook that takes the final output and saves it to a Markdown file on your local file system. You do not need to write any code in this next cell.

In [7]:
def save_file_hook(result):
    """
    Save the final research report to a local markdown file
    """
    try:
        # Get the final report content from the last task output
        if hasattr(result, 'tasks_output') and result.tasks_output:
            report_content = result.tasks_output[-1].raw
        else:
            report_content = str(result)
        
        filename = f"research_report.md"
        
        # Save to file
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(report_content)
        
        print(f"Report successfully saved to: {filename}")
        
    except Exception as e:
        print(f"Error saving report to file: {str(e)}")

<a id="7"></a>

## 7. Crew

<a id="7-1"></a>

### 7.1. Define the crew
Now you are ready to define the crew to run the deep research. As with the previous lab, you will need to define the agents and tasks. This time, you will also add the after kickoff hook and memory to the Crew.

To add the execution hook, you need to set the argument `after_kickoff_callbacks` with a list containing all the after kickoff hooks you need to run, in this case the `save_file_hook`/

In [8]:
# Create the urban planning crew
deep_research_crew = Crew(
    # include all the agents
    agents=[research_planner, 
            internet_researcher, 
            fact_checker, 
            report_writer],
    # include all the tasks in the order to be executed
    tasks=[create_research_plan, 
           gather_research_data, 
           verify_information_quality, 
           write_final_report],

    ### START CODE HERE ###
    
    # add memory to the crew
    memory=True,
    # add the after kickoff hook
    after_kickoff_callbacks=[save_file_hook]

    ### END CODE HERE ###
)

<a id="7-2"></a>

### 7.2. Define the inputs

Use the next cell to define the inputs to your Crew. This should represent the user's query. Write your own query, what would you like information about?

In [9]:
### START CODE HERE ###

# write your query in the "user_query" value
inputs = { 
    "user_query": "Create a summary of important historic events that happened in the United States during the lifetime of author Edgar Allan Poe."
}

### END CODE HERE ###

<a id="7-3"></a>

### 7.3. Run the crew
Now you can run, or kick off, the crew to get the result.

In [10]:
# Execute the crew's tasks
result = deep_research_crew.kickoff(inputs=inputs)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Task: Break down the research query "Create a summary of important historic events that happened in the        │
│  United States during the lifetime of author Edgar Allan Poe." into specific topics and key questions that      │
│  need investigation. Create a focused research plan.                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Research Plan: Important Historic Events in the United States During the Lifetime of Edgar Allan Poe**       │
│                                                                                                                 │
│  1. **Main Research Topics**                                                                                    │
│     a. **Life Background of Edgar Allan Poe**                                                                   │
│     b. **Significant National Events (1809-1849)**                                                              │
│     c. **Social and Cultural Movements**                                                                        │
│     d. **Impact of the War of 1812 and Other Conflicts**                                                        │
│     e. **Political Developments**                                                                               │
│     f. **Scientific and Technological Advances**                                                                │
│     g. **Artistic Movements and Literary Landscape**                                                            │
│                                                                                                                 │
│  2. **Key Questions for Each Topic**                                                                            │
│     a. **Life Background of Edgar Allan Poe**                                                                   │
│        - What were the key events in Poe's early life and education?                                            │
│        - How did personal tragedies influence his work and worldview?                                           │
│        - What major influences shaped Poe's literary career?                                                    │
│                                                                                                                 │
│     b. **Significant National Events (1809-1849)**                                                              │
│        - What major events occurred in the United States from 1809 to 1849?                                     │
│        - How did these events impact American society and culture?                                              │
│        - What role did Poe play, if any, in reacting to these events in his writings?                           │
│                                                                                                                 │
│     c. **Social and Cultural Movements**                                                                        │
│        - What were the principal social reforms and cultural movements during Poe's lifetime?                   │
│        - How did movements such as abolitionism, women's rights, and education reform manifest during this      │
│  period?                                                                                                        │
│        - How did Poe's works reflect or respond to these cultural dynamics?                                     │
│                                                                                                                 │
│     d. **Impact of the War of 1812 and Other Conflicts**                                                        │
│        - How did the War of 1812 affect American societ

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Task: Using the research plan, search the internet and scrape relevant websites to collect comprehensive       │
│  information on all identified topics. Verify information across multiple sources and cite all sources used.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Thought: To begin my research on the significant historic events in the United States during the lifetime of   │
│  Edgar Allan Poe, I'll start with the first research topic: the life background of Edgar Allan Poe.             │
│                                                                                                                 │
│  Using Tool: EXASearchTool                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "search_query": "Life background of Edgar Allan Poe",                                                        │
│    "start_published_date": null,                                                                                │
│    "end_published_date": null,                                                                                  │
│    "include_domains": null                                                                                      │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  The following text is scraped website content:                                                                 │
│  Timeline of the history of the United States (1820–1859) - Wikipedia                                           │
│  Jump to content                                                                                                │
│  Main menu                                                                                                      │
│  Main menu                                                                                                      │
│  move to sidebar                                                                                                │
│  hide                                                                                                           │
│  Navigation                                                                                                     │
│  Main page Contents Current events Random article About Wikipedia Contact us                                    │
│  Contribute                                                                                                     │
│  Help Learn to edit Community portal Recent changes Upload file Special pages                                   │
│  Search                                                                                                         │
│  Search                                                                                                         │
│  Appearance                                                                                                     │
│  Donate                                                                                                         │
│  Create account                                                                                                 │
│  Log in                                                                                                         │
│  Personal tools                                                                                                 │
│  Donate Create account Log in                                                                                   │
│  Contents                                                                                                       │
│  move to sidebar                                                                                                │
│  hide                                                                                                           │
│  (Top)                                                                                                          │
│  1                                                                                                              │
│  1820s                                                                                                          │
│  Toggle 1820s subsection                                                                                        │
│  1.1                                                                                                            │
│  Presidency of James Monroe                                                                                     │
│  1.2                                                                                                            │
│  Presidency of John Quincy Adams                                                                                │
│  1.3                                                                                                            │
│  Presidency of Andrew Jackson                                                                                   │
│  2                                                    

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact Checker                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Quality Assessment Report: Significant National Events in the United States During the Lifetime of Edgar     │
│  Allan Poe**                                                                                                    │
│                                                                                                                 │
│  1. **Original Data Collected**                                                                                 │
│     Events and Context:                                                                                         │
│     - The Louisiana Purchase (1803) doubled the size of the U.S., opening vast territories for settlement.      │
│     - The War of 1812 (1812-1815) affirmed U.S. sovereignty and impacted national identity, with major battles  │
│  such as the Battle of New Orleans in 1815.                                                                     │
│     - The Monroe Doctrine (1823) established a foreign policy declaring opposition to European colonialism in   │
│  the Americas.                                                                                                  │
│     - The Missouri Compromise (1820) attempted to balance slave and free states, emblematic of rising tensions  │
│  regarding slavery.                                                                                             │
│     - The Indian Removal Act (1830) led to forced relocations, notably the Trail of Tears (1838-1839).          │
│     - The Mexican-American War (1846-1848) resulted in significant territorial gains for the U.S., including    │
│  California and other southwestern territories.                                                                 │
│     - The Seneca Falls Convention (1848) marked the beginning of the women's rights movement in the U.S.        │
│                                                                                                                 │
│  2. **Verified Facts vs. Questionable Information**                                                             │
│     - The events of the Louisiana Purchase, War of 1812, Monroe Doctrine, and others are well-documented and    │
│  widely accepted as significant historical milestones.                                                          │
│     - The role of Poe in responding to these events through his writings is less directly captured              │
│  historically, but themes of identity and struggle in his works reflect the national context.                   │
│                                                                                                                 │
│  3. **Consistency Check Results**                                                                               │
│     - The events retrieved are consistent with established historical timelines and accounts from various       │
│  credible sources including Wikipedia, historical databases, and academic publications. No major discrepancies  │
│  were noted across the different sources.                                                                       │
│                                                                                                                 │
│  4. **Source Reliability Ratings**                                                                              │
│     - Wikipedia (general): Good for initial discovery; 

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Task: Create a comprehensive report that answers the original query "Create a summary of important historic    │
│  events that happened in the United States during the lifetime of author Edgar Allan Poe." using all verified   │
│  research data. Structure it with clear sections, include citations, and provide actionable insights.           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ---                                                                                                            │
│                                                                                                                 │
│  # Comprehensive Research Report: Important Historic Events in the United States During the Lifetime of Edgar   │
│  Allan Poe (1809-1849)                                                                                          │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│  This report provides a detailed analysis of significant historical events that transpired in the United        │
│  States from 1809 to 1849, coinciding with the life of the eminent author, Edgar Allan Poe. Key events          │
│  explored include territorial expansion, political developments, social movements, and conflicts that shaped    │
│  the nation's identity during this transformative era. The report synthesizes how these events interacted with  │
│  Poe's life and work, offering insights into the complex narrative of American history and literature.          │
│                                                                                                                 │
│  ## Detailed Findings                                                                                           │
│                                                                                                                 │
│  ### 1. Life Background of Edgar Allan Poe                                                                      │
│  Edgar Allan Poe was born on January 19, 1809, in Boston, Massachusetts. His early life was marked by loss,     │
│  which included the abandonment by his father and the death of his mother. Orphaned by the age of two, he was   │
│  taken in by John and Frances Allan in Richmond, Virginia, although the relationship was tumultuous and deeply  │
│  affected his personal outlook. Poe's educational pursuits began at the University of Virginia, but gambling    │
│  debts forced him to leave, leading to a brief military career before dedicating himself to writing             │
│  (Wikipedia, 2025; PBS, 2006).                                                                                  │
│                                                                                                                 │
│  ### 2. Significant National Events (1809-1849)                                                                 │
│  Numerous critical events shaped the U.S. during Poe's lifetime:                                                │
│                                                                                                                 │
│  - **The Louisiana Purchase (1803):** This monumental acquisition effectively doubled the size of the United    │
│  States and allowed for westward expansion. It set the stage for continued territorial growth.                  │
│                                                                                                                 │
│  - **The War of 1812 (1812-1815):** This conflict against Britain fostered a sense of national identity and     │
│  self-reliance. The war's conclusion, marked by the dec

Report successfully saved to: research_report.md


Once the crew is done, you should be able to see the newly created Markdown file with your report in the file navigator on the left. 

Congratulations, you reached the end of this lab! 🎉